In [1]:
import json
import sys
from pathlib import Path

from config import GROQ_API_KEY, GROQ_MODEL, TOP_K, RAW_DATA_DIR, SUPPORTED_EXTENSIONS
from loaders import load_documents, list_document_paths
from chunking import create_chunks
from embeddings import EmbeddingModel
from vector_store import create_vector_store
from retriever import retrieve, format_retrieved_chunks
from chains import classify_query, build_rag_answer, run_question
from output_parser import build_not_found_output, save_output_with_timestamp, save_output_to_json
from logger import log_query


c:\Users\Rohit kumar\Desktop\ai_knowledge_assistant_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

if not GROQ_API_KEY:
    print("ERROR: GROQ_API_KEY is not set. Create a .env file with your Groq API key.")
    sys.exit(1)

print(f"GROQ Model: {GROQ_MODEL}")
print(f"Top-K: {TOP_K}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Supported extensions: {SUPPORTED_EXTENSIONS}")

docs = list_document_paths()
print(f"\nDocuments found: {len(docs)}")
for d in docs:
    print(f"  - {d.name}")

GROQ Model: llama-3.3-70b-versatile
Top-K: 5
Raw data directory: C:\Users\Rohit kumar\Desktop\ai_knowledge_assistant_rag\data\raw
Supported extensions: ['.pdf', '.txt', '.md']

Documents found: 1
  - Amazon-2025-Annual-Report.pdf


In [3]:
# Load documents
print("=" * 60)
print("LOADING DOCUMENTS")
print("=" * 60)

documents = load_documents()
print(f"Loaded {len(documents)} document page(s).")
for doc in documents[:3]:
    print(f"  - {doc['source_file']}, Page {doc['page_number']} ({len(doc['text'])} chars)")

LOADING DOCUMENTS
Loaded 91 document page(s).
  - Amazon-2025-Annual-Report.pdf, Page 1 (21 chars)
  - Amazon-2025-Annual-Report.pdf, Page 2 (4146 chars)
  - Amazon-2025-Annual-Report.pdf, Page 3 (5022 chars)


In [4]:
# Chunk documents
print("=" * 60)
print("CHUNKING DOCUMENTS")
print("=" * 60)

chunks = create_chunks(documents)
print(f"Created {len(chunks)} chunks.")
if chunks:
    print(f"First chunk preview: {chunks[0]['text'][:100]}...")

CHUNKING DOCUMENTS
Created 410 chunks.
First chunk preview: ANNUAL REPORT
2 0 2 5...


In [5]:
# Generate embeddings
print("=" * 60)
print("GENERATING EMBEDDINGS")
print("=" * 60)

embedding_model = EmbeddingModel("sentence-transformers/all-MiniLM-L6-v2")
texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.embed_documents(texts)

print(f"Model: sentence-transformers/all-MiniLM-L6-v2")
print(f"Generated {len(embeddings)} embeddings")
if embeddings:
    print(f"Embedding dimension: {len(embeddings[0])}")

GENERATING EMBEDDINGS


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5359.37it/s]


Model: sentence-transformers/all-MiniLM-L6-v2
Generated 410 embeddings
Embedding dimension: 384


In [6]:
# Create vector store
print("=" * 60)
print("CREATING VECTOR STORE")
print("=" * 60)

vector_store = create_vector_store(chunks, embeddings)
print(f"Vector store type: {type(vector_store).__name__}")
print(f"Vector store ready.")

CREATING VECTOR STORE
Vector store type: ChromaVectorStore
Vector store ready.


In [7]:
# Classify query type
print("=" * 60)
print("CLASSIFYING QUERY TYPE")
print("=" * 60)

question = "What does Amazon say about AWS and AI-related growth?"
print(f"QUESTION: {question}")

query_data = classify_query(question)
print(f"\nQuery Type: {query_data['query_type']}")
print(f"Answer Style: {query_data['answer_style']}")

CLASSIFYING QUERY TYPE
QUESTION: What does Amazon say about AWS and AI-related growth?

Query Type: FACTUAL_LOOKUP
Answer Style: direct


In [8]:
# Retrieve top-k chunks
print("=" * 60)
print("RETRIEVING TOP-K CHUNKS")
print("=" * 60)

retrieved_chunks = retrieve(question, vector_store, embedding_model, top_k=TOP_K)
print(f"Retrieved {len(retrieved_chunks)} chunks.")
print()
print(format_retrieved_chunks(retrieved_chunks))

RETRIEVING TOP-K CHUNKS
Retrieved 5 chunks.

Rank 1 | Score: 0.7021
  Source: Amazon-2025-Annual-Report.pdf | Page: 5 | Chunk: Amazon-2025-Annual-Report.pdf_p005_003_23b201ef79
  Preview: ond, as
customers expand their use of AI, they want their inference to reside near their other applications and data
(for latency reasons), and much more of it resides in A WS than anywhere else. Thir...

Rank 2 | Score: 0.7087
  Source: Amazon-2025-Annual-Report.pdf | Page: 5 | Chunk: Amazon-2025-Annual-Report.pdf_p005_001_e317f9ba7c
  Preview: 1/ We have never seen a technology more quickly adopted than AI. When ChatGPT launched in November 2022,
it reached 100 million users in two months—four times faster than TikT ok and 15 times faster t...

Rank 3 | Score: 0.7414
  Source: Amazon-2025-Annual-Report.pdf | Page: 8 | Chunk: Amazon-2025-Annual-Report.pdf_p008_002_f062000bb3
  Preview: m for what’s ahead.
Our retail business is now approaching $600 billion in topline, yet roughly 80% of global retail

In [9]:
# Generate answer
print("=" * 60)
print("GENERATING ANSWER")
print("=" * 60)

if not retrieved_chunks:
    output = build_not_found_output(question, query_data["query_type"], TOP_K)
    output["query_type"] = query_data["query_type"]
else:
    output = build_rag_answer(question, retrieved_chunks, GROQ_API_KEY, GROQ_MODEL)
    output["query_type"] = query_data["query_type"]

log_query(output)

print(f"\nANSWER:")
print(f"  {output['answer']}")
print(f"\nAnswerability: {output['answerability']}")
print(f"Confidence: {output['confidence']}")

GENERATING ANSWER

ANSWER:
  Amazon states that AWS is experiencing significant growth due to the increasing adoption of AI, with customers choosing AWS for their AI needs. AWS has the broadest and most capable offerings, and its strong security and operational performance make it an attractive choice for customers. As a result, AWS is growing rapidly, with a $142 billion revenue run rate and 24% year-over-year growth in Q4 2025.

Answerability: Answerable
Confidence: High


In [10]:
# Display sources
print("=" * 60)
print("SOURCES")
print("=" * 60)
if output.get("sources"):
    for src in output["sources"]:
        sf = src.get('source_file', 'N/A') if isinstance(src, dict) else src
        sp = src.get('page_number', 'N/A') if isinstance(src, dict) else ''
        print(f"  - File: {sf}, Page: {sp}")
        if isinstance(src, dict) and src.get('snippet'):
            print(f"    Snippet: {src['snippet'][:150]}...")

SOURCES
  - File: Amazon-2025-Annual-Report.pdf | Page: 5 | Chunk: Amazon-2025-Annual-Report.pdf_p005_003_23b201ef79, Page: 
  - File: Amazon-2025-Annual-Report.pdf | Page: 5 | Chunk: Amazon-2025-Annual-Report.pdf_p005_001_e317f9ba7c, Page: 
  - File: Amazon-2025-Annual-Report.pdf | Page: 8 | Chunk: Amazon-2025-Annual-Report.pdf_p008_002_f062000bb3, Page: 


In [11]:
# Full JSON output
print("=" * 60)
print("FULL OUTPUT (JSON)")
print("=" * 60)
print(json.dumps(output, indent=2, ensure_ascii=False))

FULL OUTPUT (JSON)
{
  "question": "What does Amazon say about AWS and AI-related growth?",
  "query_type": "FACTUAL_LOOKUP",
  "answer": "Amazon states that AWS is experiencing significant growth due to the increasing adoption of AI, with customers choosing AWS for their AI needs. AWS has the broadest and most capable offerings, and its strong security and operational performance make it an attractive choice for customers. As a result, AWS is growing rapidly, with a $142 billion revenue run rate and 24% year-over-year growth in Q4 2025.",
  "supporting_evidence": "Customers are expanding their use of AI, and AWS has the broadest and most capable offerings. AWS has the strongest security and operational performance of any AI and infrastructure provider. AWS added 3.9 gigawatts of new power capacity in 2025 and expects to double total power capacity by the end of 2027.",
  "sources": [
    {
      "source_file": "Amazon-2025-Annual-Report.pdf | Page: 5 | Chunk: Amazon-2025-Annual-Report

In [12]:
# Save JSON output to file
print("=" * 60)
print("SAVING OUTPUT TO JSON")
print("=" * 60)
saved_path = save_output_with_timestamp(output)
print(f"Saved to: {saved_path}")


SAVING OUTPUT TO JSON
Saved to: C:\Users\Rohit kumar\Desktop\ai_knowledge_assistant_rag\outputs\output_20260612_114729.json


In [14]:
# Test with a speculative question
print("=" * 60)
print("TEST: SPECULATIVE QUESTION")
print("=" * 60)

speculative_q = "What will Tesla’s stock price be next year??"
print(f"QUESTION: {speculative_q}")

qdata = classify_query(speculative_q)
print(f"Query Type: {qdata['query_type']}")
print(f"Answer Style: {qdata['answer_style']}")


out = run_question(speculative_q)

print(f"\nANSWER:")
print(f"  {out['answer']}")
print(f"\nAnswerability: {out['answerability']}")
print(f"Confidence: {out['confidence']}")

spec_path = save_output_with_timestamp(out)
print(f"\nSaved speculative output to: {spec_path}")

print(f"\nFULL OUTPUT (JSON):")
print(json.dumps(out, indent=2, ensure_ascii=False))

TEST: SPECULATIVE QUESTION
QUESTION: What will Tesla’s stock price be next year??
Query Type: UNANSWERABLE_OR_SPECULATIVE
Answer Style: refusal


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4349.84it/s]



ANSWER:
  I could not find this information in the provided documents. The question appears to be speculative or ask about future events, which cannot be answered from the available document collection.

Answerability: NOT_FOUND
Confidence: LOW

Saved speculative output to: C:\Users\Rohit kumar\Desktop\ai_knowledge_assistant_rag\outputs\output_20260612_125553.json

FULL OUTPUT (JSON):
{
  "question": "What will Tesla’s stock price be next year??",
  "query_type": "UNANSWERABLE_OR_SPECULATIVE",
  "answer": "I could not find this information in the provided documents. The question appears to be speculative or ask about future events, which cannot be answered from the available document collection.",
  "supporting_evidence": [],
  "sources": [],
  "confidence": "LOW",
  "answerability": "NOT_FOUND",
  "retrieval_debug": {
    "top_k": 5,
    "retrieved_chunks": []
  }
}
